# 🏆 Pretraining 4 MoE Architectures on Kaggle (2x NVIDIA T4 GPUs)
**Reference Paper:** *Coupling Experts and Routers in Mixture-of-Experts via an Auxiliary Loss* (ICLR 2026)

### 4 Mô hình đối chứng hoàn toàn công bằng (Fair Benchmark):
Cả 4 mô hình đều dùng chung: **110.29M tham số (Active ~35.4M)**, 8 layers, $d=512, D=256$, context length 512, tối ưu AdamW $(\beta_1=0.9, \beta_2=0.95, \text{wd}=0.1)$, Cosine LR ($4\text{e-}4 \to 4\text{e-}5$), stream cùng tập dữ liệu `dolma-v1.5-sample`:
1. **`erc` (MoE + ERC Loss):** Đề xuất của bài báo ICLR 2026, khớp Router và Expert qua hàm Auxiliary Loss $O(n^2)$.
2. **`vanilla` (Vanilla MoE):** Kiến trúc Switch MoE truyền thống với Load Balancing Loss.
3. **`aoe` (Autonomy-of-Experts):** Chuẩn mực từ bài báo gốc `2647_Autonomy_of_Experts_Model.pdf` (Lv et al., ICML 2025) với ma trận phân rã $W_{\text{down}}$, định tuyến qua chuẩn kích hoạt và Softmax Top-K.
4. **`deepseek` (DeepSeek-MoE):** Kiến trúc DeepSeek-MoE (Dai et al., 2024) gồm 1 Shared Expert (luôn kích hoạt) + 15 Routed Experts (chọn Top-1).

### 1. Kiểm tra 2 GPU T4 của Kaggle

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} (VRAM: {torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB)")

### 2. Cài đặt các thư viện & Đăng nhập WandB

In [ ]:
!pip install -q transformers datasets accelerate einops matplotlib wandb

import os
import wandb

# Tự động đọc WANDB_API_KEY từ Kaggle Secrets (hỗ trợ Save Version / Run in Background)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = wandb_key
    wandb.login(key=wandb_key)
    print('✅ Đăng nhập WandB thành công qua Kaggle Secrets!')
except Exception as e:
    print('Kaggle Secrets chưa được thiết lập. Hãy nhập key trực tiếp nếu chạy interactive:')
    wandb.login()


### 3. [Model 1] MoE + ERC Loss (Đề xuất chính của Paper ICLR 2026)

In [ ]:
!torchrun --nproc_per_node=2 entrypoint/train.py \
    --model_type erc \
    --dolma_num_shards 30 \
    --seq_len 512 \
    --batch_size 8 \
    --grad_accum_steps 4 \
    --lr 4e-4 \
    --min_lr 4e-5 \
    --weight_decay 0.1 \
    --beta1 0.9 \
    --beta2 0.95 \
    --alpha 1.0 \
    --erc_weight 1.0 \
    --max_steps 4000 \
    --save_every 500 \
    --use_wandb \
    --wandb_project moe-erc-iclr2026 \
    --wandb_run_name erc_moe_110m \
    --output_dir /kaggle/working/checkpoints_erc

### 4. [Model 2] Vanilla MoE (Baseline Switch Transformer)

In [ ]:
!torchrun --nproc_per_node=2 entrypoint/train.py \
    --model_type vanilla \
    --dolma_num_shards 30 \
    --seq_len 512 \
    --batch_size 8 \
    --grad_accum_steps 4 \
    --lr 4e-4 \
    --min_lr 4e-5 \
    --weight_decay 0.1 \
    --beta1 0.9 \
    --beta2 0.95 \
    --max_steps 4000 \
    --save_every 500 \
    --use_wandb \
    --wandb_project moe-erc-iclr2026 \
    --wandb_run_name vanilla_moe_110m \
    --output_dir /kaggle/working/checkpoints_vanilla

### 5. [Model 3] AoE - Autonomy-of-Experts (Lv et al., ICML 2025)

In [ ]:
!torchrun --nproc_per_node=2 entrypoint/train.py \
    --model_type aoe \
    --dolma_num_shards 30 \
    --seq_len 512 \
    --batch_size 8 \
    --grad_accum_steps 4 \
    --lr 4e-4 \
    --min_lr 4e-5 \
    --weight_decay 0.1 \
    --beta1 0.9 \
    --beta2 0.95 \
    --max_steps 4000 \
    --save_every 500 \
    --use_wandb \
    --wandb_project moe-erc-iclr2026 \
    --wandb_run_name aoe_110m \
    --output_dir /kaggle/working/checkpoints_aoe

### 6. [Model 4] DeepSeek-MoE (Dai et al., 2024 - Shared + Routed Experts)

In [ ]:
!torchrun --nproc_per_node=2 entrypoint/train.py \
    --model_type deepseek \
    --dolma_num_shards 30 \
    --seq_len 512 \
    --batch_size 8 \
    --grad_accum_steps 4 \
    --lr 4e-4 \
    --min_lr 4e-5 \
    --weight_decay 0.1 \
    --beta1 0.9 \
    --beta2 0.95 \
    --max_steps 4000 \
    --save_every 500 \
    --use_wandb \
    --wandb_project moe-erc-iclr2026 \
    --wandb_run_name deepseek_moe_110m \
    --output_dir /kaggle/working/checkpoints_deepseek

### 7. So sánh kết quả cả 4 mô hình & Xuất biểu đồ báo cáo

In [ ]:
import json
import os
import matplotlib.pyplot as plt

paths = {
    'MoE + ERC Loss (Ours)': ('/kaggle/working/checkpoints_erc/training_history.json', '#d95f02', '-', 2.5),
    'Vanilla MoE': ('/kaggle/working/checkpoints_vanilla/training_history.json', '#2b5c8f', '--', 2.0),
    'AoE (Lv et al., 2025)': ('/kaggle/working/checkpoints_aoe/training_history.json', '#7570b3', ':', 2.0),
    'DeepSeek-MoE (Dai et al., 2024)': ('/kaggle/working/checkpoints_deepseek/training_history.json', '#1b9e77', '-.', 2.0)
}

plt.figure(figsize=(11, 5.5))
for label, (p, color, style, width) in paths.items():
    if os.path.exists(p):
        with open(p, 'r') as f:
            d = json.load(f)
        plt.plot([x['step'] for x in d], [x['loss'] for x in d], label=label, color=color, linestyle=style, linewidth=width)

plt.title('Fair Benchmark: Pretraining 4 MoE Architectures from Scratch on Dolma v1.5', fontsize=12, fontweight='bold')
plt.xlabel('Training Steps', fontsize=11)
plt.ylabel('Cross-Entropy Task Loss', fontsize=11)
plt.legend(fontsize=10.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/fair_comparison_4_models.png', dpi=200)
plt.show()
print('Saved plot to /kaggle/working/fair_comparison_4_models.png')

### 8. [Evaluation] Đánh giá toàn diện 10 Benchmark trong bài báo ICLR 2026

Chạy kiểm thử 4 mô hình MoE đối chứng trên toàn bộ các bộ dataset từ bài báo gốc:
1. **Language Modeling PPL**: Wikitext-2 (đo perplexity của mô hình hóa ngôn ngữ).
2. **10 Downstream Core Reasoning Benchmarks (Section 4.1 & Figure 9)**:
   - `HellaSwag`: Commonsense NLI & narrative continuation reasoning.
   - `ARC-Challenge`: Thử thách suy luận khoa học cấp độ cao.
   - `SciQ`: Bộ câu hỏi khoa học vật lý, hóa học, sinh học.
   - `OpenBookQA`: Đọc hiểu và tra cứu tri thức mở.
   - `BoolQ`: Đọc hiểu văn bản dạng True/False.
   - `WinoGrande`: Giải quyết liên kết đại từ ngữ nghĩa (Commonsense Coreference).
   - `CommonsenseQA` (C-QA): Suy luận logic tri thức đời sống hằng ngày.
   - `COPA`: Suy luận quan hệ nguyên nhân - kết quả (Causal Reasoning).
   - `Social IQa` (Social-IQa): Trí thông minh cảm xúc và tình huống xã hội.
   - `MMLU`: Bài thi trắc nghiệm tri thức đa ngành.
3. **Inference Speed**: Đo thông lượng thực tế (`tokens/sec`) và độ trễ (`ms/token`).

*Lưu ý:* Hệ thống hỗ trợ tự động tìm kiếm weights trong `/kaggle/working` hoặc `/kaggle/input` (nếu bạn load checkpoint từ Kaggle Dataset).

In [ ]:
# Tự động phát hiện thư mục chứa checkpoints (ở /kaggle/working hoặc /kaggle/input)
import os
ckpt_dir = '/kaggle/working'
if not any(os.path.exists(os.path.join(ckpt_dir, f'checkpoints_{m}')) for m in ['erc', 'vanilla', 'aoe', 'deepseek']):
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if any(f.endswith('.pt') for f in files):
                ckpt_dir = root
                break
print(f'Sử dụng Checkpoint Dir: {ckpt_dir}')

# Chạy toàn diện 10 Core Benchmarks của bài báo + Log thẳng lên WandB
!python entrypoint/evaluate.py \
    --checkpoints_dir {ckpt_dir} \
    --output_dir /kaggle/working/eval_results \
    --models erc,vanilla,aoe,deepseek \
    --tasks all \
    --max_samples -1 \
    --eval_speed \
    --use_wandb \
    --wandb_project moe-erc-iclr2026-eval \
    --wandb_run_name fair_benchmark_4_models


In [ ]:
import os
from IPython.display import display, Markdown, Image

md_path = '/kaggle/working/eval_results/evaluation_summary.md'
img_path = '/kaggle/working/eval_results/benchmark_comparison.png'

if os.path.exists(md_path):
    with open(md_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

if os.path.exists(img_path):
    display(Image(img_path))
